# AI-Powered Student Performance Prediction (Google Colab)
Run cells top to bottom. Uses **synthetic** data only. The file-writing cells make this notebook fully self-contained.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib streamlit pytest

In [ ]:
import os
os.makedirs('studentperf',exist_ok=True); os.makedirs('data/raw',exist_ok=True); os.makedirs('tests',exist_ok=True)

In [ ]:
%%writefile config.py
"""Central configuration: columns, ranges, seeds."""
RANDOM_SEED = 42
TEST_SIZE = 0.2
VAL_SIZE = 0.2          # fraction of the remaining train data
PASS_MARK = 50          # final_score >= PASS_MARK -> passed
AT_RISK_THRESHOLD = 0.5 # predicted pass probability below this -> flagged

ID_COL = "student_id"
TARGET_SCORE = "final_score"
TARGET = "passed"

# column -> (min, max) for numeric fields
FEATURE_RANGES = {
    "attendance_pct": (0, 100),
    "study_hours_week": (0, 80),
    "prior_gpa": (0, 10),
    "assignment_avg": (0, 100),
    "midterm_score": (0, 100),
    "participation_score": (0, 10),
}
FEATURES = list(FEATURE_RANGES)
REQUIRED_COLUMNS = [ID_COL] + FEATURES + [TARGET_SCORE]



In [ ]:
%%writefile generate_data.py
"""Generate the SYNTHETIC sample dataset (no real students). Run: python generate_data.py"""
import numpy as np, pandas as pd
from config import RANDOM_SEED

def make(n=600, seed=RANDOM_SEED):
    r = np.random.default_rng(seed)
    ability = r.normal(0, 1, n)
    df = pd.DataFrame({
        "student_id": [f"S{i:04d}" for i in range(1, n + 1)],
        "attendance_pct": np.clip(78 + 9 * ability + r.normal(0, 8, n), 20, 100),
        "study_hours_week": np.clip(10 + 3 * ability + r.normal(0, 3, n), 0, 50),
        "prior_gpa": np.clip(6.8 + 1.0 * ability + r.normal(0, .6, n), 2, 10),
        "assignment_avg": np.clip(68 + 10 * ability + r.normal(0, 7, n), 10, 100),
        "midterm_score": np.clip(62 + 14 * ability + r.normal(0, 9, n), 5, 100),
        "participation_score": np.clip(6 + 1.5 * ability + r.normal(0, 1.2, n), 0, 10),
    }).round(1)
    df["final_score"] = np.clip(
        0.35 * df.midterm_score + 0.25 * df.assignment_avg + 0.15 * df.attendance_pct
        + 3 * df.prior_gpa + 0.4 * df.study_hours_week + r.normal(0, 5, n) - 12, 0, 100).round(1)
    # inject realistic data-quality problems for the quality checks to find
    df.loc[r.choice(n, 15, replace=False), "attendance_pct"] = np.nan
    df.loc[r.choice(n, 10, replace=False), "midterm_score"] = np.nan
    df = pd.concat([df, df.sample(5, random_state=1)], ignore_index=True)  # duplicates
    return df

if __name__ == "__main__":
    make().to_csv("data/raw/students_synthetic.csv", index=False)
    print("wrote data/raw/students_synthetic.csv")



In [ ]:
%%writefile studentperf/__init__.py
"""Core logic for the student performance prediction project."""


In [ ]:
%%writefile studentperf/validation.py
"""Input validation: returns (errors, warnings); never raises on bad user data."""
import pandas as pd
from config import REQUIRED_COLUMNS, FEATURE_RANGES, TARGET_SCORE, ID_COL

def load_csv(file_or_path, name="uploaded"):
    """Read a CSV and return (df, errors). Records the source in df.attrs."""
    try:
        df = pd.read_csv(file_or_path)
    except pd.errors.EmptyDataError:
        return None, ["The file is empty."]
    except Exception as e:
        return None, [f"Could not read file as CSV: {e}"]
    if df.empty:
        return None, ["The file has headers but no rows."]
    df.attrs["source"] = str(name)
    return df, []

def validate(df):
    errors, warnings = [], []
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        return [f"Missing required columns: {missing}"], warnings
    for c in list(FEATURE_RANGES) + [TARGET_SCORE]:
        coerced = pd.to_numeric(df[c], errors="coerce")
        bad = coerced.isna() & df[c].notna()
        if bad.any():
            errors.append(f"Column '{c}' has {int(bad.sum())} non-numeric value(s).")
            continue
        lo, hi = FEATURE_RANGES.get(c, (0, 100))
        out = ((coerced < lo) | (coerced > hi)).sum()
        if out:
            warnings.append(f"Column '{c}': {int(out)} value(s) outside [{lo}, {hi}] (will be dropped).")
    if len(df) < 50:
        warnings.append("Fewer than 50 rows: results will be unreliable.")
    return errors, warnings



In [ ]:
%%writefile studentperf/preparation.py
"""Deterministic data preparation with a log of every transformation."""
import pandas as pd
from config import *

def prepare(df):
    """Return (clean_df, log). Same input -> same output."""
    log = []
    d = df.copy()
    n0 = len(d)
    d[ID_COL] = d[ID_COL].astype(str).str.strip()
    for c in FEATURES + [TARGET_SCORE]:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    n = len(d); d = d.drop_duplicates(); log.append(f"Removed {n - len(d)} duplicate rows.")
    n = len(d); d = d.drop_duplicates(subset=ID_COL, keep="first"); log.append(f"Removed {n - len(d)} rows with repeated student_id.")
    n = len(d); d = d.dropna(subset=[TARGET_SCORE]); log.append(f"Dropped {n - len(d)} rows with missing final_score (target).")
    mask = pd.Series(True, index=d.index)
    for c, (lo, hi) in FEATURE_RANGES.items():
        mask &= d[c].isna() | d[c].between(lo, hi)
    mask &= d[TARGET_SCORE].between(0, 100)
    log.append(f"Dropped {int((~mask).sum())} rows with out-of-range values.")
    d = d[mask]
    log.append(f"Missing feature values left as NaN here; median-imputed inside the model pipeline "
               f"(fit on training data only): {int(d[FEATURES].isna().sum().sum())} cells.")
    d[TARGET] = (d[TARGET_SCORE] >= PASS_MARK).astype(int)
    log.append(f"Created target 'passed' = final_score >= {PASS_MARK}.")
    log.append(f"Rows: {n0} -> {len(d)}.")
    return d.reset_index(drop=True), log



In [ ]:
%%writefile studentperf/model.py
"""Model logic (no UI code). Baseline = Logistic Regression; comparison = Random Forest."""
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix)
from config import *

def split(df):
    tr, te = train_test_split(df, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=df[TARGET])
    tr, va = train_test_split(tr, test_size=VAL_SIZE, random_state=RANDOM_SEED, stratify=tr[TARGET])
    return tr, va, te

def build(kind="logreg"):
    est = (LogisticRegression(max_iter=1000, random_state=RANDOM_SEED) if kind == "logreg"
           else RandomForestClassifier(n_estimators=200, max_depth=6, random_state=RANDOM_SEED))
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("m", est)])

def metrics(model, df):
    X, y = df[FEATURES], df[TARGET]
    p = model.predict(X); pr = model.predict_proba(X)[:, 1]
    out = dict(accuracy=accuracy_score(y, p), precision=precision_score(y, p, zero_division=0),
               recall=recall_score(y, p, zero_division=0), f1=f1_score(y, p, zero_division=0))
    out["roc_auc"] = roc_auc_score(y, pr) if y.nunique() > 1 else float("nan")
    return out, confusion_matrix(y, p, labels=[0, 1])

def train_and_evaluate(df):
    """Fit both models on train; report validation + test metrics; return everything needed by the UI."""
    tr, va, te = split(df)
    res = {"splits": (len(tr), len(va), len(te)), "models": {}}
    for kind in ("logreg", "rf"):
        m = build(kind).fit(tr[FEATURES], tr[TARGET])
        vm, _ = metrics(m, va); tm, cm = metrics(m, te)
        res["models"][kind] = dict(model=m, val=vm, test=tm, cm=cm)
    res["test_df"] = te
    return res

def importances(model, kind="logreg"):
    est = model.named_steps["m"]
    vals = est.coef_[0] if kind == "logreg" else est.feature_importances_
    return pd.Series(vals, index=FEATURES).sort_values(key=abs, ascending=False)

def predict_one(model, row: dict):
    X = pd.DataFrame([row])[FEATURES]
    return float(model.predict_proba(X)[0, 1])

def predict_all(model, df):
    out = df[[ID_COL]].copy()
    out["pass_probability"] = model.predict_proba(df[FEATURES])[:, 1].round(3)
    out["at_risk"] = out["pass_probability"] < AT_RISK_THRESHOLD
    return out

def error_analysis(model, te, k=8):
    """Most confidently wrong test predictions."""
    d = te.copy(); d["pass_probability"] = model.predict_proba(te[FEATURES])[:, 1]
    d["predicted"] = (d.pass_probability >= .5).astype(int)
    w = d[d.predicted != d[TARGET]].copy()
    w["confidence_error"] = (w.pass_probability - w[TARGET]).abs()
    return w.sort_values("confidence_error", ascending=False).head(k)[[ID_COL] + FEATURES + [TARGET_SCORE, TARGET, "pass_probability"]]



In [ ]:
%%writefile tests/test_core.py
import sys, os, io
sys.path.insert(0, os.path.dirname(os.path.dirname(__file__)))
import numpy as np, pandas as pd, pytest
from generate_data import make
from studentperf.validation import load_csv, validate
from studentperf.preparation import prepare
from studentperf import model as M
from config import *

@pytest.fixture(scope="module")
def raw(): return make()

def test_sample_validates(raw):
    e, _ = validate(raw); assert e == []

def test_missing_column_detected(raw):
    e, _ = validate(raw.drop(columns=["prior_gpa"])); assert e and "prior_gpa" in e[0]

def test_non_numeric_detected(raw):
    d = raw.copy(); d["attendance_pct"] = d["attendance_pct"].astype(object); d.loc[0, "attendance_pct"] = "abc"
    e, _ = validate(d); assert any("non-numeric" in x for x in e)

def test_out_of_range_warns(raw):
    d = raw.copy(); d.loc[0, "attendance_pct"] = 150
    _, w = validate(d); assert any("outside" in x for x in w)

def test_empty_file():
    df, e = load_csv(io.StringIO("")); assert df is None and e

def test_headers_only_file():
    df, e = load_csv(io.StringIO(",".join(REQUIRED_COLUMNS) + "\n")); assert df is None and e

def test_duplicates_removed(raw):
    c, _ = prepare(raw); assert not c.duplicated().any() and c[ID_COL].is_unique

def test_out_of_range_rows_dropped(raw):
    d = raw.copy(); d.loc[0, "attendance_pct"] = 150
    c, _ = prepare(d); assert c[FEATURES].max().le([FEATURE_RANGES[f][1] for f in FEATURES]).all()

def test_prepare_is_repeatable(raw):
    a, _ = prepare(raw); b, _ = prepare(raw); pd.testing.assert_frame_equal(a, b)

def test_target_definition(raw):
    c, _ = prepare(raw); assert ((c[TARGET_SCORE] >= PASS_MARK).astype(int) == c[TARGET]).all()

def test_model_beats_majority_baseline(raw):
    c, _ = prepare(raw); r = M.train_and_evaluate(c)
    maj = max(c[TARGET].mean(), 1 - c[TARGET].mean())
    assert r["models"]["logreg"]["test"]["accuracy"] > maj - 0.02
    assert r["models"]["logreg"]["test"]["roc_auc"] > 0.8

def test_training_is_reproducible(raw):
    c, _ = prepare(raw)
    a = M.train_and_evaluate(c)["models"]["logreg"]["test"]; b = M.train_and_evaluate(c)["models"]["logreg"]["test"]
    assert a == b

def test_predict_one_handles_missing_and_range(raw):
    c, _ = prepare(raw); m = M.train_and_evaluate(c)["models"]["logreg"]["model"]
    row = {f: (lo + hi) / 2 for f, (lo, hi) in FEATURE_RANGES.items()}; row["attendance_pct"] = np.nan
    assert 0 <= M.predict_one(m, row) <= 1

def test_good_student_scores_higher_than_weak(raw):
    c, _ = prepare(raw); m = M.train_and_evaluate(c)["models"]["logreg"]["model"]
    hi = dict(attendance_pct=95, study_hours_week=20, prior_gpa=9, assignment_avg=90, midterm_score=90, participation_score=9)
    lo = dict(attendance_pct=40, study_hours_week=2, prior_gpa=4, assignment_avg=40, midterm_score=30, participation_score=2)
    assert M.predict_one(m, hi) > M.predict_one(m, lo)



In [ ]:
%%writefile app.py
"""Streamlit UI.  Run: streamlit run app.py"""
import os, sys
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import io, pandas as pd, streamlit as st, matplotlib.pyplot as plt
from config import *
from studentperf.validation import load_csv, validate
from studentperf.preparation import prepare
from studentperf import model as M

st.set_page_config(page_title="Student Performance Prediction", layout="wide")
st.title("AI-Powered Student Performance Prediction")
st.caption("Prototype on SYNTHETIC / anonymised data only. Predictions are decision support, not verdicts about any student.")

@st.cache_data(show_spinner="Preparing data...")
def _prep(df): return prepare(df)

@st.cache_resource(show_spinner="Training models...")
def _train(df): return M.train_and_evaluate(df)

# ---- 1. Input
st.sidebar.header("1. Data")
up = st.sidebar.file_uploader("Upload CSV", type=["csv"])
use_sample = st.sidebar.checkbox("Use bundled synthetic sample", value=up is None)
if up is not None:
    raw, errs = load_csv(up, up.name)
elif use_sample:
    raw, errs = load_csv("data/raw/students_synthetic.csv", "students_synthetic.csv")
else:
    st.info("Upload a CSV or tick the sample option."); st.stop()
if errs:
    for e in errs: st.error(e)
    st.stop()
errors, warnings = validate(raw)
for e in errors: st.error(e)
for w in warnings: st.warning(w)
if errors: st.stop()
st.sidebar.success(f"Source: {raw.attrs['source']} ({len(raw)} rows)")

clean, log = _prep(raw)
if len(clean) < 30 or clean[TARGET].nunique() < 2:
    st.error("Not enough usable rows (or only one outcome class) after cleaning to train a model."); st.stop()
res = _train(clean)
key = st.sidebar.selectbox("Model", ["logreg", "rf"], format_func=lambda k: {"logreg": "Logistic Regression (baseline)", "rf": "Random Forest (comparison)"}[k])
mdl = res["models"][key]

t1, t2, t3, t4, t5 = st.tabs(["Data quality", "Trends", "Evaluation", "Predict a student", "Cohort & export"])

with t1:
    st.subheader("Preparation log")
    for l in log: st.write("•", l)
    st.dataframe(raw.isna().sum().rename("missing values"))

with t2:
    st.subheader("Aggregate trends")
    c1, c2 = st.columns(2)
    f = c1.selectbox("Feature", FEATURES)
    fig, ax = plt.subplots(); clean.boxplot(column=f, by=TARGET, ax=ax); ax.set_xlabel("passed (0=no, 1=yes)"); plt.suptitle("")
    c1.pyplot(fig)
    fig, ax = plt.subplots(); clean[FEATURES + [TARGET_SCORE]].corr()[TARGET_SCORE].drop(TARGET_SCORE).sort_values().plot.barh(ax=ax)
    ax.set_title("Correlation with final_score"); c2.pyplot(fig)

with t3:
    st.subheader("Evaluation (held-out test set)")
    st.caption(f"Train/val/test sizes: {res['splits']}. Seed={RANDOM_SEED}. Metrics are from synthetic data and say nothing about real-world accuracy.")
    st.dataframe(pd.DataFrame({k: {"val " + a: b for a, b in v["val"].items()} | {"test " + a: b for a, b in v["test"].items()}
                               for k, v in res["models"].items()}).round(3))
    c1, c2 = st.columns(2)
    fig, ax = plt.subplots(); ax.imshow(mdl["cm"], cmap="Blues")
    for (i, j), v in __import__("numpy").ndenumerate(mdl["cm"]): ax.text(j, i, v, ha="center", va="center")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1]); ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (test)")
    c1.pyplot(fig)
    imp = M.importances(mdl["model"], key)
    fig, ax = plt.subplots(); imp.sort_values().plot.barh(ax=ax)
    ax.set_title("Coefficient (standardised)" if key == "logreg" else "Feature importance"); c2.pyplot(fig)
    st.write("**Most confident errors on test set**")
    st.dataframe(M.error_analysis(mdl["model"], res["test_df"]))
    st.caption("Typical causes: students near the pass mark, unusual combinations (e.g. high attendance but low scores), and imputed missing values.")

with t4:
    st.subheader("Individual prediction (enter values manually)")
    vals = {}
    cols = st.columns(3)
    for i, (c, (lo, hi)) in enumerate(FEATURE_RANGES.items()):
        vals[c] = cols[i % 3].number_input(c, float(lo), float(hi), float((lo + hi) / 2), key=c)
    if st.button("Predict"):
        p = M.predict_one(mdl["model"], vals)
        st.metric("Estimated pass probability", f"{p:.0%}")
        (st.warning if p < AT_RISK_THRESHOLD else st.success)("Flagged for support follow-up" if p < AT_RISK_THRESHOLD else "Not flagged")

with t5:
    preds = M.predict_all(mdl["model"], clean)
    st.metric("Students flagged at-risk", int(preds.at_risk.sum()), f"of {len(preds)}")
    st.dataframe(preds.sort_values("pass_probability").head(50))
    st.download_button("Download predictions CSV", preds.to_csv(index=False), "predictions.csv", "text/csv")
    st.download_button("Download cleaned data CSV", clean.to_csv(index=False), "cleaned.csv", "text/csv")



## 1. Create / load dataset
Generates the synthetic CSV. To use your own, upload a CSV with the required columns (see README data dictionary) and point `read_csv` at it.

In [ ]:
!python generate_data.py
import pandas as pd
raw = pd.read_csv('data/raw/students_synthetic.csv')
raw.head()

## 2. Validate and prepare

In [ ]:
from studentperf.validation import validate
from studentperf.preparation import prepare
errors, warnings = validate(raw)
print('errors:', errors, '\nwarnings:', warnings)
print(raw.isna().sum())
clean, log = prepare(raw)
print('\n'.join(log))

## 3. Trends

In [ ]:
import matplotlib.pyplot as plt
from config import *
clean[FEATURES+[TARGET_SCORE]].corr()[TARGET_SCORE].drop(TARGET_SCORE).sort_values().plot.barh(title='Correlation with final_score')
plt.show()
clean.groupby(TARGET)[FEATURES].mean().round(1)

## 4. Train and evaluate (baseline Logistic Regression vs Random Forest)

In [ ]:
from studentperf import model as M
res = M.train_and_evaluate(clean)
print('train/val/test sizes:', res['splits'])
for k,v in res['models'].items():
    print(k, 'TEST', {a: round(b,3) for a,b in v['test'].items()})
    print('confusion matrix [[TN,FP],[FN,TP]]:\n', v['cm'])

In [ ]:
M.importances(res['models']['logreg']['model']).sort_values().plot.barh(title='Logistic regression standardised coefficients')
plt.show()
M.error_analysis(res['models']['logreg']['model'], res['test_df'])

## 5. Predictions and export

In [ ]:
m = res['models']['logreg']['model']
preds = M.predict_all(m, clean)
preds.to_csv('predictions.csv', index=False)
print(preds.at_risk.sum(), 'flagged of', len(preds))
print('Single student pass probability:', M.predict_one(m, dict(attendance_pct=60, study_hours_week=5, prior_gpa=5.5, assignment_avg=55, midterm_score=45, participation_score=4)))
# from google.colab import files; files.download('predictions.csv')

## 6. Tests

In [ ]:
!python -m pytest -q tests

## 7. (Optional) Launch the Streamlit app inside Colab
Run the cell, open the printed URL, and enter the tunnel password shown (your public IP).

In [ ]:
!npm -q install localtunnel
!streamlit run app.py --server.port 8501 &>/content/logs.txt &
import urllib.request, time; time.sleep(5)
print('Tunnel password (IP):', urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
!npx localtunnel --port 8501